# Coal ReaxFF bonds：分子片段与泛粗粒度分析

本 Notebook 读取 `bond3512.reaxff`，提取分子片段，计算元素组成、质量、环数和自由基候选位点，并生成泛粗粒度节点统计与 SVG 可视化。

In [ ]:
from pathlib import Path
import sys
import json
from collections import Counter

cwd = Path.cwd()
if (cwd / 'coal_reaction_network' / 'src').exists():
    project_dir = cwd / 'coal_reaction_network'
elif (cwd / 'src').exists() and (cwd / 'data').exists():
    project_dir = cwd
else:
    project_dir = next((p / 'coal_reaction_network' for p in [cwd, *cwd.parents] if (p / 'coal_reaction_network' / 'src').exists()), None)
    if project_dir is None:
        raise FileNotFoundError('请从仓库根目录或 coal_reaction_network 目录启动 Notebook。')

SRC_DIR = project_dir / 'src'
DATA_DIR = project_dir / 'data'
RESULT_DIR = project_dir / 'results' / 'bond3512'
BONDS_FILE = DATA_DIR / 'bond3512.reaxff'
sys.path.insert(0, str(SRC_DIR))
assert BONDS_FILE.exists(), f'输入文件不存在: {BONDS_FILE}'
print('项目目录:', project_dir)
print('输入文件:', BONDS_FILE)
print('输出目录:', RESULT_DIR)

## 1. 读取 bonds 文件并提取片段

默认映射为 `1=C, 2=H, 3=O`，有效键阈值为 `bond_order >= 0.3`。

In [ ]:
from reaxff_fragments import ReaxFFBondFile, write_outputs
from pseudo_species import ThreeDimensionalClassifier

ELEMENT_MAP = {1: 'C', 2: 'H', 3: 'O'}
source = ReaxFFBondFile(BONDS_FILE, element_map=ELEMENT_MAP, bond_order_cutoff=0.3).parse()
classifier = ThreeDimensionalClassifier()
fragments = source.extract_fragments(classifier=classifier)
json_path, csv_path = write_outputs(source, fragments, RESULT_DIR)
print('timestep:', source.timestep)
print('原子数:', len(source.atoms))
print('有效键数:', len(source.edges))
print('分子片段数:', len(fragments))
print('实际泛粗粒度节点数:', len(classifier.node_definitions()))
print('JSON:', json_path)
print('CSV :', csv_path)

In [ ]:
all_atom_ids = [atom_id for fragment in fragments for atom_id in fragment.atom_ids]
assert len(all_atom_ids) == len(source.atoms)
assert len(all_atom_ids) == len(set(all_atom_ids))
assert sum(len(fragment.atom_details) for fragment in fragments) == len(source.atoms)
print('atom ID 覆盖检查通过；唯一 atom ID 数:', len(set(all_atom_ids)))

## 2. 节点分类统计

理论节点数为 `5 × 5 × 4 = 100`；当前帧只会实际出现其中一部分。

In [ ]:
node_counts = Counter(fragment.node_id for fragment in fragments)
oxygen_counts = Counter(fragment.oxygen_bin for fragment in fragments)
print('氧分类分布:', dict(oxygen_counts))
for node_id, count in node_counts.most_common():
    item = next(fragment for fragment in fragments if fragment.node_id == node_id)
    print(f'{node_id:>3} | {count:>4} fragments | {item.carbon_bin}, {item.ring_bin}, {item.oxygen_bin}')

In [ ]:
largest = sorted(fragments, key=lambda item: len(item.atom_ids), reverse=True)[:10]
for item in largest:
    print({
        'fragment_id': item.fragment_id, 'node_id': item.node_id,
        'atoms': len(item.atom_ids), 'elements': item.element_counts,
        'mass': round(item.molecular_mass, 3), 'rings': item.ring_count,
        'radicals': item.radical_sites, 'first_atom_ids': item.atom_ids[:12]
    })

## 3. 当前帧可视化

生成四个氧类别面板的碳数×环数节点网格，以及片段质量—自由基属性散点图。

In [ ]:
from visualize_frame import write_node_grid, write_fragment_scatter
with json_path.open(encoding='utf-8') as handle:
    result_data = json.load(handle)
node_grid_path = RESULT_DIR / 'bond3512_fragments_node_grid.svg'
scatter_path = RESULT_DIR / 'bond3512_fragments_fragment_scatter.svg'
write_node_grid(result_data, node_grid_path)
write_fragment_scatter(result_data, scatter_path)
print(node_grid_path)
print(scatter_path)

In [ ]:
try:
    from IPython.display import SVG, display
    display(SVG(filename=str(node_grid_path)))
    display(SVG(filename=str(scatter_path)))
except ImportError:
    print('SVG 已生成，请用浏览器或 VS Code 打开。')

## 4. 数据驱动聚类分析

本节不是 100 类预定义节点统计，而是对真实片段的 `log1p(C)、log1p(环)、log1p(O)` 特征执行 K-means 聚类。

In [ ]:
from cluster_fragments import analyze_clusters, write_cluster_svg
# 本 Notebook 前面的提取单元会生成 json_path；若从别处运行，则自动使用默认结果文件。
FRAGMENTS_JSON = Path(json_path) if 'json_path' in globals() else RESULT_DIR / 'bond3512_fragments.json'
data = json.loads(FRAGMENTS_JSON.read_text(encoding='utf-8'))
cluster_result = analyze_clusters(data, k=None)
cluster_json_path = RESULT_DIR / 'bond3512_fragments_clusters.json'
cluster_svg_path = RESULT_DIR / 'bond3512_fragments_cluster_distribution.svg'
cluster_json_path.write_text(json.dumps(cluster_result, ensure_ascii=False, indent=2), encoding='utf-8')
write_cluster_svg(cluster_result, cluster_svg_path)
print('selected K:', cluster_result['selected_k'])
print('聚类结果:', cluster_json_path)
print('聚类图:', cluster_svg_path)

In [ ]:
for summary in cluster_result['cluster_summaries']:
    print(summary)
try:
    from IPython.display import SVG, display
    display(SVG(filename=str(cluster_svg_path)))
except ImportError:
    print('聚类 SVG 已生成，请用浏览器或 VS Code 打开。')

## 5. 两帧节点关系

这里直接在同一个 Notebook 中比较 `bond0.reaxff` 和 `bond19998.reaxff`。它使用 atom-ID 重叠建立 fragment lineage，再投影成粗粒度节点之间的有向关系。

In [ ]:
from fragment_lineage import compare_bonds_files, write_transition_outputs
from visualize_transitions import write_transition_svg

TRANSITION_DIR = project_dir / 'results' / 'transitions'
PARENT_FILE = DATA_DIR / 'bond0.reaxff'
CHILD_FILE = DATA_DIR / 'bond19998.reaxff'
transition_result = compare_bonds_files(PARENT_FILE, CHILD_FILE)
transition_json_path, transition_reactions_csv, transition_node_edges_csv = write_transition_outputs(transition_result, TRANSITION_DIR)
transition_svg_path = TRANSITION_DIR / f"bond{transition_result['parent_timestep']}_to_bond{transition_result['child_timestep']}_network.svg"
write_transition_svg(transition_result, transition_svg_path)
print('parent fragments:', transition_result['parent_fragment_count'])
print('child fragments :', transition_result['child_fragment_count'])
print('lineage links    :', transition_result['lineage_link_count'])
print('events           :', transition_result['event_count'])
print('node edges       :', len(transition_result['node_edges']))
print('reactions        :', len(transition_result['aggregated_reactions']))
print('json             :', transition_json_path)
print('reactions csv    :', transition_reactions_csv)
print('node edges csv   :', transition_node_edges_csv)
print('svg              :', transition_svg_path)

In [ ]:
for reaction in transition_result['aggregated_reactions'][:20]:
    print(f"{reaction['event_count']:>5}  {reaction['reaction_signature']}")
try:
    from IPython.display import SVG, display
    display(SVG(filename=str(transition_svg_path)))
except ImportError:
    print('两帧节点关系 SVG 已生成，请用浏览器或 VS Code 打开。')

## 6. 启动可视化界面

运行下面的代码单元会启动本地 dashboard，并在本单元输出中嵌入界面。若 VS Code 不允许嵌入，请点击输出中的链接，或在 VS Code 的 Ports 面板转发端口 `8765` 后打开转发地址。界面里可以手动输入 bonds 文件路径，切换单帧/两帧模式，并筛选你想看的节点。

In [ ]:
import sys, time
from urllib.request import urlopen
from IPython.display import HTML, IFrame, display
from network_dashboard import start_dashboard

DASHBOARD_HOST = '127.0.0.1'
DASHBOARD_PORT = 8765
dashboard_url = f'http://localhost:{DASHBOARD_PORT}'
dashboard_probe_url = f'http://127.0.0.1:{DASHBOARD_PORT}'

def dashboard_is_running(url):
    try:
        with urlopen(url, timeout=0.5) as response:
            return response.status == 200
    except Exception:
        return False

dashboard_server = globals().get('dashboard_server')
dashboard_thread = globals().get('dashboard_thread')
if not dashboard_is_running(dashboard_probe_url):
    if dashboard_server is not None:
        dashboard_server.shutdown()
        dashboard_server.server_close()
    dashboard_server, dashboard_thread = start_dashboard(DASHBOARD_HOST, DASHBOARD_PORT)
    for _ in range(100):
        if dashboard_is_running(dashboard_probe_url):
            break
        if not dashboard_thread.is_alive():
            dashboard_server.server_close()
            raise RuntimeError('Dashboard 后台线程已退出，请检查端口或 Python 错误。')
        time.sleep(0.1)
    else:
        dashboard_server.shutdown()
        dashboard_server.server_close()
        raise RuntimeError(f'Dashboard 未能监听端口 {DASHBOARD_PORT}，请检查 VS Code Ports 是否转发了同一个端口。')
else:
    print('检测到已有 dashboard 服务，继续复用。')

display(HTML(f"<p>可视化界面已启动：<a href='{dashboard_url}' target='_blank'>{dashboard_url}</a>（点击链接可在浏览器打开）</p>"))
display(IFrame(src=dashboard_url, width='100%', height=1100))
print('停止服务：dashboard_server.shutdown(); dashboard_server.server_close()')
